# Does US Drought Predict Agricultural Futures Returns?
### Machine Learning in Portfolio Management and Trading — ENSAE Paris, MS Finance et Gestion des Risques

*Authors: [Name 1], [Name 2]*

**Abstract.** *(to be written at the end)*

## 1. Introduction

*(to be completed)*

- **Context:** weather is a key driver of agricultural supply; the US Corn Belt accounts for a large share of world corn and soybean production.
- **Hypothesis:** the US Drought Monitor (USDM), a public weekly indicator, contains information on future corn and soybean futures returns that prices do not fully incorporate (under-reaction to public information).
- **Approach:** build a Corn Belt drought indicator, forecast weekly futures returns with simple ML models in a walk-forward setting, and evaluate the resulting trading strategy out-of-sample.
- **Related literature:** Roll (1984); Boudoukh et al. (2007); Brooks, Mattos & Schoengold (2014); Lehecka (2014); Hong, Li & Xu (2019); Kuwayama et al. (2019).

## 2. Data

This section builds a single weekly dataset combining (i) drought conditions in the main corn and soybean producing states and (ii) futures returns. Particular attention is paid to **when** each piece of information becomes publicly available, to avoid look-ahead bias.

### 2.1 Drought data: the US Drought Monitor

The **US Drought Monitor (USDM)** is produced jointly by the National Drought Mitigation Center (University of Nebraska-Lincoln), the USDA, NOAA and NASA. Each week, experts synthesise several indicators (precipitation, soil moisture, streamflow, vegetation health, etc.) into a map that classifies each area into five categories:

| Category | Meaning |
|---|---|
| D0 | Abnormally dry |
| D1 | Moderate drought |
| D2 | Severe drought |
| D3 | Extreme drought |
| D4 | Exceptional drought |

**Drought Severity and Coverage Index (DSCI).** The USDM summarises a map into a single number:

$$\text{DSCI} = 1\cdot D0 + 2\cdot D1 + 3\cdot D2 + 4\cdot D3 + 5\cdot D4$$

where $Dk$ is the *categorical* percentage of the area in category $k$. The DSCI ranges from 0 (no dryness) to 500 (whole area in D4).

**Timing (critical for look-ahead bias).** Each map reflects conditions as of **Tuesday** morning (the *valid date*, `MapDate` in the data) but is only **published on Thursday**. The data are indexed by the valid date, so they must be re-indexed to the publication date before being matched with prices.

**Access.** Statistics are available through a public REST API (no key required), by state, from January 2000.

**Scope.** We use the main corn and soybean producing states. States are identified in the API by their two-digit FIPS code.

#### 2.1.1 Setup and parameters

All parameters (dates, states, folders) are defined once here and reused in the rest of the notebook.

In [1]:
import os
import time
from io import StringIO

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

# Folder where all downloaded data are cached
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

# Sample period
START_DATE = pd.Timestamp("2000-01-01")
END_DATE = pd.Timestamp.today().normalize()

# Main corn and soybean producing states -> two-digit FIPS codes (used by the USDM API)
STATES = {
    "IA": "19", "IL": "17", "NE": "31", "MN": "27", "IN": "18", "SD": "46",
    "OH": "39", "WI": "55", "KS": "20", "MO": "29", "ND": "38",
}

USDM_BASE_URL = "https://usdmdataservices.unl.edu/api/StateStatistics"

#### 2.1.2 Downloading USDM statistics by state

We download two series for each state:
- the **categorical** percentage of area in each drought category (None, D0 to D4), which allows us to recompute the DSCI ourselves;
- the **DSCI** as published by the USDM, used as a cross-check.

Downloads are cached as CSV files in the `data/` folder, so the notebook can be re-run without calling the API again.

In [2]:
def usdm_date(d):
    """Format a date as M/D/YYYY, the format expected by the USDM API."""
    return f"{d.month}/{d.day}/{d.year}"


def fetch_usdm_state(fips, endpoint, statistics_type):
    """Download USDM statistics for one state and return them as a DataFrame."""
    url = (f"{USDM_BASE_URL}/{endpoint}?aoi={fips}"
           f"&startdate={usdm_date(START_DATE)}&enddate={usdm_date(END_DATE)}"
           f"&statisticsType={statistics_type}")
    response = requests.get(url, headers={"Accept": "text/csv"}, timeout=120)
    response.raise_for_status()
    return pd.read_csv(StringIO(response.text))


def load_usdm(endpoint, statistics_type, name):
    """Download all states, or read the cached CSV if it already exists."""
    path = os.path.join(DATA_DIR, f"usdm_{name}.csv")
    if os.path.exists(path):
        print(f"Reading cached file {path}")
        return pd.read_csv(path)
    frames = []
    for state, fips in STATES.items():
        df = fetch_usdm_state(fips, endpoint, statistics_type)
        df["state"] = state
        frames.append(df)
        print(f"{state}: {len(df)} rows")
        time.sleep(0.5)  # be polite with the public API
    out = pd.concat(frames, ignore_index=True)
    out.to_csv(path, index=False)
    return out


# Categorical % of area in None, D0..D4 (statisticsType=2)
usdm_cat_raw = load_usdm("GetDroughtSeverityStatisticsByAreaPercent", 2, "categorical")
# Official DSCI, used as a cross-check
usdm_dsci_raw = load_usdm("GetDSCI", 1, "dsci")

IA: 1398 rows
IL: 1398 rows
NE: 1398 rows
MN: 1398 rows
IN: 1398 rows
SD: 1398 rows
OH: 1398 rows
WI: 1398 rows
KS: 1398 rows
MO: 1398 rows
ND: 1398 rows
IA: 1398 rows
IL: 1398 rows
NE: 1398 rows
MN: 1398 rows
IN: 1398 rows
SD: 1398 rows
OH: 1398 rows
WI: 1398 rows
KS: 1398 rows
MO: 1398 rows
ND: 1398 rows


#### 2.1.3 First inspection of the raw data

In [3]:
for name, df in [("Categorical", usdm_cat_raw), ("DSCI", usdm_dsci_raw)]:
    print(f"--- {name}: {df.shape[0]} rows")
    print(df.dtypes)
    display(df.head(3))

print(usdm_cat_raw.groupby("state").size())

--- Categorical: 15378 rows
MapDate                int64
StateAbbreviation        str
None                 float64
D0                   float64
D1                   float64
D2                   float64
D3                   float64
D4                   float64
ValidStart               str
ValidEnd                 str
StatisticFormatID      int64
state                    str
dtype: object


,MapDate,StateAbbreviation,None,D0,D1,D2,D3,D4,ValidStart,ValidEnd,StatisticFormatID,state
0,20261013,IA,98.64,1.36,0.00,0.0,0.0,0.0,2026-10-13,2026-10-19,2,IA
1,20261006,IA,98.64,1.36,0.00,0.0,0.0,0.0,2026-10-06,2026-10-12,2,IA
2,20260929,IA,91.49,8.47,0.04,0.0,0.0,0.0,2026-09-29,2026-10-05,2,IA


--- DSCI: 15378 rows
Name         str
MapDate    int64
DSCI       int64
state        str
dtype: object


,Name,MapDate,DSCI,state
0,Iowa,20000104,215,IA
1,Iowa,20000111,217,IA
2,Iowa,20000118,217,IA


state
IA    1398
IL    1398
IN    1398
KS    1398
MN    1398
MO    1398
ND    1398
NE    1398
OH    1398
SD    1398
WI    1398
dtype: int64


#### 2.1.4 Cleaning and consistency checks

*(to be completed)*

#### 2.1.5 From valid date to publication date

*(to be completed)*

#### 2.1.6 Corn Belt aggregate indicators (production-weighted)

*(to be completed)*

### 2.2 Futures prices

*(to be completed)*

### 2.3 Final weekly dataset

*(to be completed)*

## 3. Feature engineering

*(to be completed)*

## 4. Exploratory analysis

*(to be completed)*

## 5. Learning and backtest

*(to be completed)*

## 6. Baseline results

*(to be completed)*

## 7. Robustness

*(to be completed)*

## 8. Conclusion

*(to be completed)*

## References

- Boudoukh, J., Richardson, M., Shen, Y. & Whitelaw, R. (2007). Do asset prices reflect fundamentals? Freshly squeezed evidence from the OJ market. *Journal of Financial Economics*.
- Brooks, Mattos & Schoengold (2014). How do agricultural futures prices respond to new information about drought conditions? *NCCC-134 Conference*.
- Hong, H., Li, F. W. & Xu, J. (2019). Climate risks and market efficiency. *Journal of Econometrics*.
- Kuwayama, Y. et al. (2019). Estimating the impact of drought on agriculture using the U.S. Drought Monitor. *American Journal of Agricultural Economics*.
- Lehecka, G. (2014). The value of USDA crop progress and condition information. *Journal of Agricultural and Resource Economics*.
- Roll, R. (1984). Orange juice and weather. *American Economic Review*.
- Svoboda, M. et al. (2002). The Drought Monitor. *Bulletin of the American Meteorological Society*.
- Welch, I. & Goyal, A. (2008). A comprehensive look at the empirical performance of equity premium prediction. *Review of Financial Studies*.